# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### The rule in one sentence

**Rank every content item by how many independent staleness-or-underperformance signals it shows
right now, and send a human the ones that show the most.**

That is the whole rule. It is deliberately a counting rule, not a weighted model: no coefficient is
fitted to anything, so every point in a row's score traces back to one named condition that a
reviewer can check by hand against the row. This is the baseline ML-08 has to beat, and a baseline
is only useful if it is honest and readable, so readability is chosen over cleverness everywhere.

### What the rule is allowed to see

The ML-04 contract (`work/outputs/data_contract.json`, version 1.1) fixes the decision date at
`T = 2026-03-31`, the feature window at `[T-89, T]`, and the label window at `[T+1, T+30]`. The rule
scores using feature-window fields only. Two protections are structural rather than promised:

1. the three label fields (`trend_direction`, `trend_pct`, `is_declining_label`) are **dropped from
   the frame before scoring** and re-attached only after every score is final;
2. the contract's excluded fields (`provider_used`, `model_used`) are dropped entirely, because they
   record which AI product touched the page and are a product decision rather than an observation
   about the content.

So the leakage claim in section 4 is a claim about which columns physically existed in the scored
frame, which is checkable, not a claim about intent.

### The reason codes

Points are round numbers on purpose. They encode a stated ordering of concern, nothing more, and no
attempt is made to defend the exact gaps between them.

| Reason code | Points | Fires when | What it is a proxy for | How it could be wrong |
|---|---|---|---|---|
| `stale_visible_page` | 30 | `days_since_last_update >= 180` and `impressions_90d >= 500` | A page people still find, that nobody has touched in half a year | Some content is reference material that is correct while old. Age is not decay. |
| `thin_visible_page` | 25 | `0 < word_count < 1200` and `impressions_90d >= 250` | Demand arriving at a page too short to satisfy it | Word count is a bad proxy for completeness. A precise 800-word answer can beat a padded 2,000-word one. |
| `page_one_decay_risk` | 20 | `0 < avg_position <= 10` and `content_age_days >= 180` | A ranking worth defending, on content old enough to be overtaken | Holding page one for a year is evidence the page is *working*, not that it is at risk. This is the most speculative code here. |
| `low_ctr_visible_page` | 15 | `impressions_90d >= 500`, `0 < avg_position <= 20`, `ctr < 0.5` | Visible in results, but the title or snippet is not earning the click | CTR is strongly intent-dependent. Informational queries answered in the SERP itself have a low CTR by nature, not by fault. |
| `low_engagement_visible_page` | 10 | `sessions_90d >= 30` and (`0 < engagement_rate < 30` or `0 < scroll_rate < 30`) | Traffic arrives and then leaves | Both rates are measurement artefacts as much as behaviour, and short pages legitimately produce low scroll. |

A row matching nothing scores 0 and is labelled `general_refresh_review`, whose action is
`monitor_only`. Ties break on `impressions_90d`, so where the evidence is equal the rule prefers the
page more people actually see.

### From reason code to action

The action is the most specific reason that applies, taken in this fixed order, so the output is one
instruction rather than a list a human has to arbitrate:

`thin_visible_page` -> `expand_and_refresh` · `stale_visible_page` -> `refresh_content` ·
`low_ctr_visible_page` -> `refresh_and_review_ctr` · `page_one_decay_risk` ->
`refresh_to_defend_position` · `low_engagement_visible_page` -> `review_engagement` ·
otherwise `monitor_only`.

### Reproducibility

The rule is deterministic and contains no sampling, so there is no seed to fix. Re-running
`work/scripts/baseline_action_score.py` on the same feature vector reproduces the CSV byte for byte.

In [1]:
# Section 1 - load the contract and state the rule as executable settings.
import json
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def find_repo_root() -> Path:
    """Resolve the repo root from Colab, the notebook folder, or the repo root itself."""
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/FlyRank-Machine-Learning-Internship"),
    ]
    for base in candidates:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "work" / "scripts"))

CONTRACT = json.loads((ROOT / "work" / "outputs" / "data_contract.json").read_text(encoding="utf-8"))
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

print(f"repo root       : {ROOT}")
print(f"contract        : v{CONTRACT['version']} (card {CONTRACT['card']})")
print(f"decision date T : {CONTRACT['windows']['decision_date']}")
print(f"feature window  : {CONTRACT['windows']['feature_window'][0]} -> {CONTRACT['windows']['feature_window'][1]}")
print(f"label window    : {CONTRACT['windows']['label_window'][0]} -> {CONTRACT['windows']['label_window'][1]}")

# The rule may never see these. Enforced in section 2, asserted in section 4.
LABEL_FIELDS = ["trend_direction", "trend_pct", "is_declining_label"]
EXCLUDED_FIELDS = CONTRACT["fields"]["starter"]["excluded"]
print(f"\nlabel fields    : {LABEL_FIELDS}")
print(f"excluded fields : {EXCLUDED_FIELDS}  (from the contract, not chosen here)")

# The windows must not touch. Restated here so this notebook fails loudly if the
# contract is ever edited into an overlapping state.
assert CONTRACT["windows"]["label_window"][0] > CONTRACT["windows"]["decision_date"], \
    "label window must start strictly after the decision date"
print("\n[OK] label window starts strictly after T")

repo root       : D:\Flyrank\FlyRank-Machine-Learning-Internship
contract        : v1.1 (card ML-04)
decision date T : 2026-03-31
feature window  : 2026-01-01 -> 2026-03-31
label window    : 2026-04-01 -> 2026-04-30

label fields    : ['trend_direction', 'trend_pct', 'is_declining_label']
excluded fields : ['provider_used', 'model_used']  (from the contract, not chosen here)

[OK] label window starts strictly after T


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The scoring logic lives in `work/scripts/baseline_action_score.py` rather than in this cell, for the
reason `work/README.md` gives: the reference pipeline in `scripts/` stays pristine, and anything I
modify is a copy under `work/`. Keeping it in a module also means ML-08 can import the identical
baseline instead of a re-typed approximation of it, which is the usual way a baseline comparison
quietly stops being a comparison.

In [2]:
# Section 2 - build the ranked queue and write the CSV.
from baseline_action_score import RULES, build, LABEL_FIELDS as MOD_LABELS, EXCLUDED_FIELDS as MOD_EXCLUDED

raw = pd.read_csv(FEATURE_PATH)
print(f"feature vector : {raw.shape[0]:,} rows x {raw.shape[1]} columns")

# Hold the labels aside BEFORE scoring. This is the leakage guard, and it is the
# reason section 4 can make a structural claim.
held_out = raw[[c for c in MOD_LABELS if c in raw.columns]].copy()
features = raw.drop(columns=[c for c in MOD_LABELS + MOD_EXCLUDED if c in raw.columns])
print(f"visible to rule: {features.shape[1]} columns "
      f"({raw.shape[1] - features.shape[1]} withheld)")

queue = build(features)
queue = queue.join(held_out.reindex(queue.index))

KEEP = ["rank", "content_id", "client_id", "action_score", "reason_codes", "reason_count",
        "suggested_action", "impressions_90d", "clicks_90d", "sessions_90d", "ctr",
        "avg_position", "word_count", "content_age_days", "days_since_last_update",
        "engagement_rate", "scroll_rate"] + [f"pts_{c}" for c, _, _ in RULES]

OUT_CSV = ROOT / "work" / "outputs" / "baseline_action_score.csv"
queue[KEEP].to_csv(OUT_CSV, index=False)
assert OUT_CSV.exists(), "queue CSV was not written"
print(f"\nwrote          : {OUT_CSV.relative_to(ROOT)} ({OUT_CSV.stat().st_size:,} bytes)")
print(f"queue length   : {len(queue):,}")

print("\nScore distribution (points -> rows):")
dist = queue["action_score"].value_counts().sort_index(ascending=False)
for score, n in dist.items():
    print(f"  {score:>3}  {n:>6,}  {'#' * max(1, int(n / 250))}")

print("\nSuggested actions:")
for action, n in queue["suggested_action"].value_counts().items():
    print(f"  {action:<28} {n:>6,}  ({n / len(queue):.1%})")

print("\nHow often each reason code fires:")
for code, points, _ in RULES:
    n = int(queue[f"pts_{code}"].gt(0).sum())
    print(f"  {code:<30} {points:>3} pts  {n:>6,}  ({n / len(queue):.2%})")

feature vector : 30,000 rows x 52 columns
visible to rule: 47 columns (5 withheld)



wrote          : work\outputs\baseline_action_score.csv (4,442,212 bytes)
queue length   : 30,000

Score distribution (points -> rows):
   65       3  #
   60       5  #
   55       8  #
   50       2  #
   45   1,245  ####
   40      13  #
   35   2,150  ########
   30     570  ##
   25   1,889  #######
   20   3,113  ############
   15   4,510  ##################
   10   2,848  ###########
    0  13,644  ######################################################

Suggested actions:
  monitor_only                 13,644  (45.5%)
  refresh_and_review_ctr        9,731  (32.4%)
  refresh_to_defend_position    3,678  (12.3%)
  review_engagement             2,848  (9.5%)
  expand_and_refresh               82  (0.3%)
  refresh_content                  17  (0.1%)

How often each reason code fires:
  stale_visible_page              30 pts      17  (0.06%)
  thin_visible_page               25 pts      82  (0.27%)
  page_one_decay_risk             20 pts   7,076  (23.59%)
  low_ctr_visible_page   

### What the distribution already says

Two things are visible before any evaluation, and both matter more than the ranking itself.

**Nearly half the corpus scores zero.** 13,644 of 30,000 rows (45.5%) match no reason code at all and
fall to `monitor_only`. That is not a queue of 30,000; it is a queue of about 16,000 with a long
inert tail, and any later claim about "the queue" has to mean the scoring half.

**Two of the five rules are effectively dead on this data.** `stale_visible_page` fires 17 times and
`thin_visible_page` 82 times, out of 30,000 rows. Both carry the highest point values, so the two
conditions I judged *most* serious when writing the rule turn out to be the two the data almost never
satisfies. The thresholds were set from domain reasoning rather than from the observed distribution,
and the distribution disagrees. Worth stating plainly rather than quietly retuning: a rule that fires
on 0.06% of rows cannot meaningfully order a queue, and the 30 points attached to it are doing almost
nothing.

The practical consequence is that the top of the queue is really decided by the three mid-weight
codes stacking, not by the two heavy ones firing.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Every row in the top 20 fires exactly three reason codes. Nothing in the corpus fires four or more,
so 65 points is the observed ceiling rather than a designed one.

In [3]:
# Section 3 - the top 20, with the evidence that put each one there.
top20 = queue.head(20).copy()

cols = ["rank", "action_score", "reason_codes", "suggested_action",
        "impressions_90d", "ctr", "avg_position", "word_count", "days_since_last_update"]
print(top20[cols].to_string(index=False))

# Confidence note: how much of the score rests on the single most speculative
# code (page_one_decay_risk), and whether any input to the score is missing.
top20["pct_from_speculative"] = top20["pts_page_one_decay_risk"] / top20["action_score"]
top20["has_missing_input"] = (top20["word_count"] <= 0) | (top20["ctr"] <= 0)

print("\nConfidence flags across the top 20:")
print(f"  rows leaning >=30% on page_one_decay_risk : {int((top20['pct_from_speculative'] >= 0.30).sum())}")
print(f"  rows with a missing or zero score input   : {int(top20['has_missing_input'].sum())}")
print(f"  distinct clients represented              : {top20['client_id'].nunique()}")

 rank  action_score                                                         reason_codes       suggested_action  impressions_90d  ctr  avg_position  word_count  days_since_last_update
    1            65          low_ctr_visible_page|page_one_decay_risk|stale_visible_page        refresh_content             1408 0.28           7.8      4758.0                     183
    2            65          low_ctr_visible_page|page_one_decay_risk|stale_visible_page        refresh_content              954 0.42           9.0      1335.0                     301
    3            65          low_ctr_visible_page|page_one_decay_risk|stale_visible_page        refresh_content              821 0.24           5.8      1504.0                     301
    4            60           low_ctr_visible_page|page_one_decay_risk|thin_visible_page     expand_and_refresh             5129 0.19           6.3      1184.0                     104
    5            60           low_ctr_visible_page|page_one_decay_risk|thin_visi

### Reading the top 20 honestly

**Ranks 1-3** (`stale_visible_page` + `page_one_decay_risk` + `low_ctr_visible_page`) are the
cleanest picks in the queue. Pages ranking 5.8-9.0, still drawing 800-1,400 impressions, untouched for
183-301 days, converting below 0.5% CTR. Three independent signals agree and none of them is
speculative. *What would make it wrong:* if these are seasonal or evergreen reference pages whose CTR
is low because the answer shows in the SERP, then nothing is broken and a refresh spends effort for
no gain.

**Ranks 4-8** swap staleness for thinness. Ranks 5, 6 and 8 were updated only 20 days ago, so
"refresh" is close to a contradiction: someone has just worked on them. The rule cannot tell the
difference between a page nobody has touched and a page someone touched last month, because
`thin_visible_page` does not look at recency at all. *What would make it wrong:* it already is, mildly.
These should sit below ranks 1-3 by more than the 5 points the current weighting gives them.

**Ranks 14 and 16 are the clearest false positives in the top 20.** Both fire `thin_visible_page`,
but rank 14 has a CTR of 5.19 at average position 2.4, and rank 16 a CTR of 2.54 at position 7.5.
Both are performing roughly an order of magnitude better than the corpus norm. The rule flagged its
best pages as needing work, purely because they are under 1,200 words. That is the word-count proxy
failing exactly the way section 1 predicted it might, and it is the strongest argument in this
notebook for replacing hand-set thresholds with a learned model in ML-08.

**Ranks 19-20 are a data-quality artefact, not a finding.** Both have `word_count == 0`, which is
missing content data, not a short page. They reach the top 20 on 517,000+ impressions and three codes
that all fire on performance fields. The rule has no concept of "input absent", so absence reads as
an extreme value. *What would make it wrong:* it is wrong now. Rows with missing inputs should be
routed to a data-repair queue rather than a content-refresh queue, and that is a change to make
before this score is shown to anyone.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Section 4 - weak picks, leakage assertions, and what the rule is actually worth.

# --- 4a. Leakage: a structural check, not an assurance ----------------------
scored_columns = set(features.columns)
for field in MOD_LABELS:
    assert field not in scored_columns, f"LEAKAGE: label field {field} was visible to the rule"
for field in MOD_EXCLUDED:
    assert field not in scored_columns, f"LEAKAGE: excluded field {field} was visible to the rule"

# No score component may correlate with the label by construction: every pts_
# column must be derivable from feature-window fields alone.
pts_cols = [f"pts_{code}" for code, _, _ in RULES]
assert all(c in queue.columns for c in pts_cols)
print("[OK] no label field reached the scorer :", MOD_LABELS)
print("[OK] no excluded field reached the scorer:", MOD_EXCLUDED)
print(f"[OK] {len(scored_columns)} columns visible, {len(pts_cols)} score components, all feature-window")

# --- 4b. Weak picks, found rather than asserted -----------------------------
top100 = queue.head(100)
missing_input = top100[top100["word_count"] <= 0]
high_performers = top100[(top100["ctr"] > 2.0) & (top100["avg_position"] <= 10)]
just_updated = top100[top100["days_since_last_update"] <= 30]

print(f"\nWeak picks inside the top 100:")
print(f"  missing word_count (data gap read as signal) : {len(missing_input):>3}")
print(f"  already high-performing (CTR>2, pos<=10)     : {len(high_performers):>3}")
print(f"  updated within the last 30 days              : {len(just_updated):>3}")
print(f"  union of the three                           : "
      f"{len(set(missing_input.index) | set(high_performers.index) | set(just_updated.index)):>3} of 100")

# --- 4c. Is the ranking worth anything? -------------------------------------
y = queue["is_declining_label"]
base_rate = float(y.mean())
print(f"\nBase rate of declining pages: {base_rate:.3f}")
print(f"{'k':>6} {'precision@k':>12} {'lift':>7}")
for k in (20, 50, 100, 500, 1000, 5000):
    p = float(queue.head(k)["is_declining_label"].mean())
    print(f"{k:>6} {p:>12.3f} {p / base_rate:>7.2f}x")

print("\nPrecision by reason code (each vs the base rate):")
for code, _, _ in RULES:
    fired = queue[queue[f"pts_{code}"] > 0]
    if len(fired):
        p = float(fired["is_declining_label"].mean())
        print(f"  {code:<30} n={len(fired):>6,}  precision={p:.3f}  lift={p / base_rate:.2f}x")

[OK] no label field reached the scorer : ['trend_direction', 'trend_pct', 'is_declining_label']
[OK] no excluded field reached the scorer: ['provider_used', 'model_used']
[OK] 47 columns visible, 5 score components, all feature-window

Weak picks inside the top 100:
  missing word_count (data gap read as signal) :  52
  already high-performing (CTR>2, pos<=10)     :   2
  updated within the last 30 days              :  55
  union of the three                           :  83 of 100

Base rate of declining pages: 0.542
     k  precision@k    lift
    20        0.650    1.20x
    50        0.680    1.25x
   100        0.620    1.14x
   500        0.548    1.01x
  1000        0.539    0.99x
  5000        0.544    1.00x

Precision by reason code (each vs the base rate):
  stale_visible_page             n=    17  precision=0.529  lift=0.98x
  thin_visible_page              n=    82  precision=0.537  lift=0.99x
  page_one_decay_risk            n= 7,076  precision=0.538  lift=0.99x
  low_ctr_v

### What the numbers say, including the part that is unflattering

**The ranking works, but only at the very top, and not for long.** Against a base rate of 0.542:

| k | precision@k | lift |
|---|---|---|
| 20 | 0.650 | 1.20x |
| 50 | 0.680 | 1.25x |
| 100 | 0.620 | 1.14x |
| 500 | 0.548 | 1.01x |
| 1,000 | 0.539 | 0.99x |

By k=500 the lift is 1.01x, and by k=1,000 it is 0.99x, which is very slightly *worse* than picking
at random. So the honest scope of this baseline is roughly **the first 100 rows**. Described as a
tool, it is "a way to find about 50-100 pages worth a human's attention", not "a way to rank 30,000
pages". Any later claim that the queue is useful at depth is not supported by anything measured here.

**No individual reason code discriminates at all.** Each of the five sits at 0.53-0.54 precision
against a 0.542 base rate, so every one of them, taken alone, is indistinguishable from random
selection. The top-of-queue lift comes entirely from *stacking* codes: rows firing three independent
signals are meaningfully more likely to be declining, while rows firing any single signal are not.

That is the most useful thing this notebook found, and it was not the expected result. It reframes
what the baseline actually is. It is not five good detectors combined; it is five weak detectors whose
*agreement* carries the signal. It also sets the bar ML-08 has to clear, and suggests where the
headroom is: a learned model can weight interactions between these signals, which is precisely the
thing a counting rule cannot express.

**83 of the top 100 are weak picks**, and this is the finding that should stop the baseline from
shipping. Measured, not estimated:

| Weakness | Rows in top 100 |
|---|---|
| `word_count` missing, so a data gap is being read as a signal | 52 |
| Updated within the last 30 days, so "refresh" is near-meaningless | 55 |
| Already high-performing (CTR > 2 at position <= 10) | 2 |
| **Union of the three** | **83** |

So the top 100, the only region where the lift table showed the rule beating random, is 83% composed
of rows a human reviewer would reject on sight. The 1.20x lift at k=20 is real, but it is being earned
*despite* the composition of the queue rather than because of it, and the two dominant faults are not
modelling problems at all: 52 rows are a missing-data problem and 55 are a staleness-definition
problem. Both are fixable without any machine learning.

This is the single most important number in the notebook and it is worse than expected. The correct
read is that the baseline is not currently a usable queue. It is a demonstration that the signals
carry *some* information at the very top, plus a list of four concrete defects to fix before anyone
sees the output.

### What I would change before this ships

1. Route rows with missing `word_count` or `ctr` to a data-repair queue instead of scoring them.
2. Add a recency guard so a page updated in the last 30 days cannot fire `thin_visible_page`.
3. Suppress the score entirely where CTR and position are both strong, whatever else fires.
4. Re-derive the `stale_visible_page` and `thin_visible_page` thresholds from the observed
   distribution, since at 17 and 82 firings they contribute almost nothing as set.

All four are stated as observations from this run and are directional, not causal: nothing here
establishes that refreshing a flagged page improves its trend, only that the flagged pages are
somewhat more likely to be declining at the top of the queue.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.